## (1) Importing modules

In [ ]:
#!/usr/bin/python
# -*- coding: UTF-8 -*-

#__modification time__ = 2026-05-10
#__author__ = Qi Zhou, Helmholtz Centre Potsdam - GFZ German Research Centre for Geosciences
#__find me__ = qi.zhou@gfz.de, qi.zhou.geo@gmail.com, https://github.com/Qi-Zhou-Geo
# Please do not distribute this code without the author's permission

import yaml

import numpy as np
import pandas as pd

from obspy import UTCDateTime

import torch
import torch.optim as optim
print("PyTorch version:", torch.__version__) # PyTorch version: 1.12.1

# region <add the sys.path to search for custom modules>
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent.parent
sys.path.append(str(project_root))


print(f"Current dir: {current_dir}\n"
      f"Project root: {project_root}")
# endregion


# import the custom functions
from functions.data_process.load_data import load_waveform_pro, clip_df_columns
from functions.data_process.dataset_to_dataloader import data_to_seq, seq_to_dataset, dataset_to_dataloader
from functions.visualize.plotly_visualize import plotly_1time_series

## (1) Load seismic features -- take 2017 as an example

### (1-1) define the parameters

In [ ]:
train_year_list = [2017, 2018, 2019]
test_year_list = [2020]

input_station_dict = {2017: 'ILL02', 2018: 'ILL12', 2019: 'ILL12', 2020: 'ILL12'}
seismic_network = "9S"
input_component = "EHZ"

### (1-2) Load the Type C features and manually label

In [ ]:
input_year = 2017
input_station = input_station_dict.get(input_year)
feature_dir = f"{project_root}/data/seismic_feature/European/Illgraben/{input_year}/{input_station}/EHZ"

# Type A set
df1 = pd.read_csv(f"{feature_dir}/{input_year}_{input_station}_{input_component}_all_A.txt",
                    header=0, low_memory=False,
                    # 14 is 'goodness', 16 is 'magnitude_range', 17 is 'alpha'
                    usecols=[4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 17])

# Type B set: waveform, spectrum, spectrogram sets
df2 = pd.read_csv(f"{feature_dir}/{input_year}_{input_station}_{input_component}_all_B.txt",
                    header=0, low_memory=False,
                    usecols=np.arange(4, 63))

assert df1.shape[0] == df2.shape[0], f"df1.shape={df1.shape[0]}, !=,  df2.shape={df2.shape[0]}"

# Type B set: network
# do NOT need this network feature, just neet it to keep the data structure
df3 = pd.DataFrame(1, index=range(df1.shape[0]), columns=range(np.arange(3, 13).size))
df3.columns = ['id_maxRMS', 'id_minRMS', 'ration_maxTOminRMS', 'ration_maxTOminIQR', 'mean_coherenceOfNet',
                'max_coherenceOfNet', 'mean_lagTimeOfNet', 'std_lagTimeOfNet', 'mean_wdOfNet', 'std_wdOfNet']


# Data label
df4 = pd.read_csv(f"{feature_dir}/{input_year}_{input_station}_{input_component}_all_A.txt",
                    header=0, low_memory=False)
amp_array = np.array(df4.iloc[:, :2])
amp_array = amp_array[:, [1, 0]]
amp_array = load_waveform_pro(amp_array, seismic_network, input_year,
                                input_station, input_component,
                                with_label=True)

df4 = pd.DataFrame(amp_array, columns=["time_stamps", "time_window_start", "label_0nonDF_1DF"])

# pack together
df = pd.concat([df1, df2, df3, df4], axis=1, ignore_index=True)
columnsName = np.concatenate([df1.columns.values, df2.columns.values, df3.columns.values, df4.columns.values])
df.columns = columnsName

print(df.shape)
print(df.columns)


### (1-3) Visualize the features

In [ ]:
column_x, column_y = 'time_window_start', 'ES_0'
# this is a pre-defined plot function, click it and check the details
fig = plotly_1time_series(df, column_x, column_y)

## Question 1: Do you see the anomaly?

### (1-4) Clip the anomaly

In [ ]:
# You might be wondering, how much data do we need to clip? Click the <clip_df_columns> function.
df = clip_df_columns(df)

column_x, column_y = 'time_window_start', 'ES_0'
fig = plotly_1time_series(df, column_x, column_y)

## Question 2: How can we ensure that the model learns from comparable data across different years, locations, and sensors?

## (2) Normalize the data

In [ ]:
# normalize the seismic features
with np.load(f"{project_root}/data/scaler/normalize_factor4C.npz", "r") as f:
    min_factor = f["min_factor"]
    max_factor = f["max_factor"]

X = df.iloc[:, :-3].to_numpy().astype(float)
scaled = (X - min_factor) / (max_factor - min_factor)
df.iloc[:, :-3] = scaled


column_x, column_y = 'time_window_start', 'ES_0'
fig = plotly_1time_series(df, column_x, column_y)

## Note: We need to transform the data according to the DL's requirements.

## (3) From df to dataloader

### (3-2) Define the parameters

In [ ]:
feature_type = "H"
seq_length = 64
batch_size = 128

### (3-1) Select the features

In [ ]:
config_path = (project_root / f"./config/config_inference.yaml").resolve()
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
    selected_column = config[f"feature_type_{feature_type}"]
selected_column.extend([80, 81, 82])

df_selected = df.iloc[:, selected_column]
print(df_selected.shape, df_selected.columns)

In [ ]:
# column of data_array as [time_stamps, features, target]
time_stamp_float, time_stamp_str = df_selected.iloc[:, -3], df_selected.iloc[:, -2]
x = df_selected.iloc[:, :-3]
y = df_selected.iloc[:, -1]
df_data_array = pd.concat([time_stamp_float, x, y], axis=1, ignore_index=True)

data_array = np.array(df_data_array)
print(data_array.shape)

In [ ]:
# chunk the 2D seismic features
train_sequences = data_to_seq(array=data_array, seq_length=seq_length)

print(data_array.shape)
print(len(train_sequences))

print(data_array.shape[0] - len(train_sequences))

print(len(train_sequences[0]))

In [ ]:
# convert the data sequences to Pytorch DataSet
dataset = seq_to_dataset(sequences=train_sequences, data_type="feature")
print(len(dataset), dataset[0]['features'].shape)


if input_year in train_year_list:
    training_or_testing = "training"
elif input_year in test_year_list:
    training_or_testing = "testing"
else:
    raise ValueError(f"input_year: {input_year}")

# convert Pytorch DataSet to Pytorch DataLoader
dataloader = dataset_to_dataloader(dataset=dataset,
                                   batch_size=batch_size,
                                   training_or_testing=training_or_testing) # "training" will shuffle the data
len(dataloader.dataset)


torch.save(dataloader, f"{current_dir}/data/{input_year}_dataloader.pt")

In [ ]:
dataloader = torch.load(f"{current_dir}/data/{input_year}_dataloader.pt")

# quickly check the data in dataloader
for batch_data in dataloader.dataLoader():
    # t_features of t to t_{sequence_length}, shape ([batch_size, sequence_length]), float time stamps
    t_features = batch_data['t_features']
    print(type(t_features), t_features.shape)
    
    # features of t to t_i, shape ([batch_size, sequence_length, num_features])
    features = batch_data['features']
    print(type(features), features.shape) 

    # t_target of t_{sequence_length+1}, shape ([batch_size, 1]), float time stamps
    t_target = batch_data['t_target']
    print(t_target.shape) 
    
    # target of t_{sequence_length+1}, shape ([batch_size, 1]), debris flow probability or label
    target = batch_data['target']
    print(target.shape)
    
    print( UTCDateTime(t_target[0].detach().cpu().numpy()) )
    
    break


## Question 3: How to deal with this?
### Count label for: ('9S', 2017, 'ILL02'),
### DF: (354,),
### Non-DF: (65886,),
### All labels: 66240

## Task 1: Can you prepare the dataloader for 2020 data?